In [3]:
from pathlib import Path 

import polars as pl 
from sqlalchemy import text 

from dbconfig import engine
print('Environment Ready!')

Environment Ready!


In [4]:
raw_dir = Path("../raw")

In [5]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS clean"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS analytics"))

In [7]:
csv_files = sorted(raw_dir.glob("*.csv"))

[f.name for f in csv_files]

[]

In [8]:
pl.read_database(
    "SELECT current_database() AS db",
    engine,)

db
str
"""olist"""


In [9]:
pl.read_database(
    """
    SELECT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
    """,
    engine,)

schema_name
str
"""analytics"""
"""clean"""
"""information_schema"""
"""pg_catalog"""
"""pg_toast"""
"""public"""
"""raw"""


In [11]:
df = pl.read_csv("raw/olist_customers_dataset.csv")

df.shape

(99441, 5)

In [12]:
df.head()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
str,str,i64,str,str
"""06b8999e2fba1a1fbc88172c00ba8b…","""861eff4711a542e4b93843c6dd7feb…",14409,"""franca""","""SP"""
"""18955e83d337fd6b2def6b18a428ac…","""290c77bc529b7ac935b93aa66c333d…",9790,"""sao bernardo do campo""","""SP"""
"""4e7b3e00288586ebd08712fdd0374a…","""060e732b5b29e8181a18229c7b0b2b…",1151,"""sao paulo""","""SP"""
"""b2b6027bc5c5109e529d4dc6358b12…","""259dac757896d24d7702b9acbbff3f…",8775,"""mogi das cruzes""","""SP"""
"""4f2d8ab171c80ec8364f7c12e35b23…","""345ecd01c38d18a9036ed96c73b8d0…",13056,"""campinas""","""SP"""


In [14]:
df.write_database(
    table_name="raw.customers",
    connection=engine,
    if_table_exists="replace",)

-1

In [16]:
pl.read_database("select count(*) from raw.customers", engine)

count
i64
99441


In [19]:
TABLE_MAP = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "payments",
    "olist_order_reviews_dataset.csv": "reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation",
}

RAW_DIR = Path("raw/")

for filename, table_name in TABLE_MAP.items():
    path = RAW_DIR / filename

    print(f"Loading {filename} -> raw.{table_name}")

    df = pl.read_csv(path)


    df.write_database(
        table_name=f"raw.{table_name}",
        connection=engine,
        if_table_exists="replace",
    )

    print(f"✓ {df.height:,} rows loaded")

Loading olist_customers_dataset.csv -> raw.customers


✓ 99,441 rows loaded
Loading olist_geolocation_dataset.csv -> raw.geolocation


✓ 1,000,163 rows loaded
Loading olist_order_items_dataset.csv -> raw.order_items


✓ 112,650 rows loaded
Loading olist_order_payments_dataset.csv -> raw.payments


✓ 103,886 rows loaded
Loading olist_order_reviews_dataset.csv -> raw.reviews


✓ 99,224 rows loaded
Loading olist_orders_dataset.csv -> raw.orders


✓ 99,441 rows loaded
Loading olist_products_dataset.csv -> raw.products


✓ 32,951 rows loaded
Loading olist_sellers_dataset.csv -> raw.sellers
✓ 3,095 rows loaded
Loading product_category_name_translation.csv -> raw.category_translation
✓ 71 rows loaded
